# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List and examine record sets
record_sets_info = []
for record_set in metadata.record_sets:
    rec_info = {
        '@id': record_set.id,
        'name': record_set.name,
        'fields': [field.id for field in record_set.fields]
    }
    record_sets_info.append(rec_info)

if record_sets_info:
    for rec in record_sets_info:
        print(f"RecordSet @id: {rec['@id']} | Name: {rec['name']} | Field @ids: {rec['fields']}")
else:
    print("No record sets found in the metadata.")

In [ ]:
# For demonstration, list available record sets and peek at their records
if metadata.record_sets:
    for record_set in metadata.record_sets:
        print(f"\nRecordSet '@id': {record_set.id}, name: {record_set.name}")
        # Print field information
        print("Fields (@id):", [field.id for field in record_set.fields])
        # Preview a few records
        try:
            # Try up to 2 records
            for i, record in enumerate(dataset.records(record_set=record_set.id)):
                print(f"Record {i+1}: {record}")
                if i >= 1:
                    break
        except Exception as e:
            print(f"Could not display records for {record_set.id}: {e}")
else:
    print("No record sets available to enumerate.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All `@id` references use the values found in the previous section.

In [ ]:
dataframes = {}

record_set_ids = [rs.id for rs in metadata.record_sets]
if record_set_ids:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    # Display columns of the first record set
    first_record_set_id = record_set_ids[0]
    print(f"Columns for record set '@id' {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()
else:
    print("No record sets found for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a record set and numeric field for analysis

# For this FAIR^2 dataset example, we attempt to find a numeric field.
# Example: Try to select record set and a numeric field heuristically.

import numpy as np

# Pick the first available record set with data
candidate_record_set_id = None
numeric_field_id = None

for rs in metadata.record_sets:
    df = dataframes.get(rs.id)
    if df is not None and not df.empty:
        # Try to find a field with numeric-appearing type
        for field in rs.fields:
            dtype = str(field.data_type).lower() if hasattr(field, 'data_type') else ''
            if ('float' in dtype or 'number' in dtype or 'integer' in dtype or 'double' in dtype or 'numeric' in dtype):
                if field.id in df.columns:
                    candidate_record_set_id = rs.id
                    numeric_field_id = field.id
                    break
        if candidate_record_set_id:
            break
# If none found, fallback to first record set and auto-detect numeric column
if candidate_record_set_id is None and record_set_ids:
    candidate_record_set_id = record_set_ids[0]
    df = dataframes[candidate_record_set_id]
    # Try to select first column with numeric dtype
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]

# Set group field as the next categorical column if available
group_field = None
if candidate_record_set_id is not None:
    df = dataframes[candidate_record_set_id]
    non_numeric_cols = [col for col in df.columns if col != numeric_field_id]
    for col in non_numeric_cols:
        if df[col].dtype == 'object':
            group_field = col
            break

if candidate_record_set_id and numeric_field_id:
    df = dataframes[candidate_record_set_id]
    # Clean numeric field (coerce errors, drop NAs)
    df = df.copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75) if df[numeric_field_id].notna().sum() > 0 else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (75th percentile):\n", filtered_df.head())
    # Normalization
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:\n", filtered_df[[numeric_field_id, col_norm]].head())
    # Grouping if possible
    if group_field is not None:
        print(f"\nGrouping by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(grouped_df.head())
else:
    print("Could not identify numeric field for EDA in the available dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Visualizing the distribution of the numeric field
import matplotlib.pyplot as plt
import seaborn as sns
if candidate_record_set_id and numeric_field_id:
    df = dataframes[candidate_record_set_id]
    # Drop missing values
    values = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()
    plt.figure(figsize=(8,4))
    sns.histplot(values, bins=30, kde=True, color='teal')
    plt.title(f'Distribution of {numeric_field_id} in record set {candidate_record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.